# CICIoV2024 – Sıralı ve rastgele bölme karşılaştırması (AS2b)

Orijinal makale (Neto vd., 2024) veriyi **her saldırı dosyası için ilk %80 eğitim / son %20 test** olarak bölmüş ve Random Forest ile makro F1 = 0,76 raporlamıştır. Sonraki çalışmalar veriyi **karıştırıp rastgele** bölerek 1,00'a ulaşmıştır.

Bu not defteri aynı modelleri, aynı veride, iki protokolle çalıştırır:
- **Sıralı bölme:** Her CSV dosyasında satır sırası korunur; ilk %80 eğitim, son %20 test.
- **Rastgele bölme:** Katmanlı, karıştırılmış %80/20 (deneyler not defterindeki AS2 ile aynı).

Ayrıca her protokolde test satırlarının yüzde kaçının eğitimde birebir bulunduğu (sızıntı) ölçülür.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/CICIoV2024/decimal'   # <-- kendi klasörünle değiştir
OUT_DIR = '/content/drive/MyDrive/CICIoV2024/sonuclar'
SEEDS = [0, 1, 2, 3, 4]
TRAIN_FRAC = 0.8

In [ ]:
import glob, os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)
os.makedirs(OUT_DIR, exist_ok=True)

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, matthews_corrcoef, accuracy_score, recall_score, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    HAS_LGB = False

# Dosyaları sırasını koruyarak oku ve her dosyada sıralı %80/20 işaretle
parts = []
for f in sorted(glob.glob(os.path.join(DATA_DIR, '*.csv'))):
    d = pd.read_csv(f)
    d.columns = [c.strip() for c in d.columns]
    cut = int(len(d) * TRAIN_FRAC)
    d['seq_split'] = np.where(np.arange(len(d)) < cut, 'train', 'test')
    d['source_file'] = os.path.basename(f)
    parts.append(d)
df = pd.concat(parts, ignore_index=True)

label_cols = [c for c in df.columns if c.lower() in ('label', 'category', 'specific_class')]
TARGET = 'specific_class' if 'specific_class' in df.columns else label_cols[-1]
feature_cols = [c for c in df.columns if c not in label_cols + ['seq_split', 'source_file']]
ID_COL = next(c for c in feature_cols if c.upper() == 'ID')
le = LabelEncoder().fit(df[TARGET])
CLASSES = list(le.classes_)
print(df.shape, '| öznitelikler:', feature_cols)
print(df.groupby(['source_file', 'seq_split']).size().unstack())

## Ortak araçlar (deneyler not defteriyle aynı)

In [ ]:
class ExactMatch:
    def fit(self, X, y):
        d = pd.DataFrame(X).assign(_y=y).drop_duplicates(subset=list(range(X.shape[1])))
        self.table = dict(zip(map(tuple, d.iloc[:, :-1].to_numpy()), d['_y']))
        vals, cnt = np.unique(y, return_counts=True)
        self.default = vals[cnt.argmax()]
        return self
    def predict(self, X):
        return np.array([self.table.get(tuple(r), self.default) for r in X])

class IDOnly:
    def __init__(self, id_idx): self.id_idx = id_idx
    def fit(self, X, y):
        self.m = DecisionTreeClassifier(random_state=0).fit(X[:, [self.id_idx]], y); return self
    def predict(self, X):
        return self.m.predict(X[:, [self.id_idx]])

id_idx = feature_cols.index(ID_COL)
def models(seed):
    m = {
        'Ezber tablosu (temel çizgi)': ExactMatch(),
        'Yalnızca CAN ID (temel çizgi)': IDOnly(id_idx),
        'Lojistik regresyon': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
        'Karar ağacı': DecisionTreeClassifier(random_state=seed),
        'Random Forest': RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=seed),
        'MLP': make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=50,
                                                             early_stopping=True, random_state=seed)),
    }
    if HAS_LGB:
        m['LightGBM'] = lgb.LGBMClassifier(n_estimators=200, random_state=seed, verbose=-1)
    return m

def leakage(tr, te):
    keys = set(map(tuple, tr[feature_cols].to_numpy()))
    seen = np.fromiter((tuple(r) in keys for r in te[feature_cols].to_numpy()), bool, len(te))
    out = te.assign(seen=seen).groupby(TARGET)['seen'].mean().mul(100).round(2)
    out['GENEL'] = round(seen.mean() * 100, 2)
    return out

## 1. Sızıntı: sıralı ve rastgele bölme

In [ ]:
seq_tr, seq_te = df[df.seq_split == 'train'], df[df.seq_split == 'test']
rnd_tr, rnd_te = train_test_split(df, test_size=1 - TRAIN_FRAC, stratify=df[TARGET], random_state=0)
leak = pd.DataFrame({'sirali_bolme_%': leakage(seq_tr, seq_te), 'rastgele_bolme_%': leakage(rnd_tr, rnd_te)})
leak.to_csv(os.path.join(OUT_DIR, 'as2b_sizinti.csv'))
leak

## 2. Modeller: sıralı ve rastgele bölme

In [ ]:
rows, recall_rows, cms = [], [], {}
for protocol in ['Sıralı', 'Rastgele']:
    for seed in SEEDS:
        if protocol == 'Sıralı':
            tr, te = seq_tr, seq_te
        else:
            tr, te = train_test_split(df, test_size=1 - TRAIN_FRAC, stratify=df[TARGET], random_state=seed)
        Xtr, Xte = tr[feature_cols].to_numpy(), te[feature_cols].to_numpy()
        ytr, yte = le.transform(tr[TARGET]), le.transform(te[TARGET])
        for name, model in models(seed).items():
            model.fit(Xtr, ytr)
            pred = model.predict(Xte)
            rows.append({'protokol': protocol, 'model': name, 'seed': seed,
                         'accuracy': accuracy_score(yte, pred),
                         'macro_F1': f1_score(yte, pred, average='macro'),
                         'MCC': matthews_corrcoef(yte, pred)})
            rec = recall_score(yte, pred, average=None, labels=range(len(CLASSES)), zero_division=0)
            recall_rows.append({'protokol': protocol, 'model': name, 'seed': seed, **dict(zip(CLASSES, rec))})
            if name == 'Random Forest' and seed == 0:
                cms[protocol] = pd.DataFrame(confusion_matrix(yte, pred, labels=range(len(CLASSES))),
                                             index=[f'gerçek_{c}' for c in CLASSES],
                                             columns=[f'tahmin_{c}' for c in CLASSES])
            print(f'{protocol:9s} seed={seed} {name:32s} macroF1={rows[-1]["macro_F1"]:.4f}')

res = pd.DataFrame(rows)
res.to_csv(os.path.join(OUT_DIR, 'as2b_ham.csv'), index=False)
ms = lambda a: f'{np.mean(a):.4f} ± {np.std(a):.4f}'
summary = res.groupby(['model', 'protokol'], sort=False)['macro_F1'].agg(ms).unstack('protokol')
summary.to_csv(os.path.join(OUT_DIR, 'as2b_ozet.csv'))
summary

## 3. Sınıf bazında duyarlılık (recall) ve Random Forest karışıklık matrisi

In [ ]:
recall = (pd.DataFrame(recall_rows).groupby(['protokol', 'model'], sort=False)[CLASSES].mean().round(3))
recall.to_csv(os.path.join(OUT_DIR, 'as2b_sinif_recall.csv'))
display(recall)
for p, cm in cms.items():
    cm.to_csv(os.path.join(OUT_DIR, f'as2b_rf_karisiklik_{"sirali" if p == "Sıralı" else "rastgele"}.csv'))
    print(f'\nRandom Forest karışıklık matrisi – {p} bölme (seed 0)')
    display(cm)

## Bana gönderilecek çıktılar
`as2b_sizinti.csv`, `as2b_ozet.csv`, `as2b_sinif_recall.csv`, `as2b_rf_karisiklik_sirali.csv`